Set database context and compare original versus revival price per ounce.
*Co-authored with CoCo*

In [ ]:
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

for weight_file in ['ClashGrotesk-Extralight.ttf', 'ClashGrotesk-Light.ttf',
                    'ClashGrotesk-Regular.ttf', 'ClashGrotesk-Medium.ttf',
                    'ClashGrotesk-Semibold.ttf', 'ClashGrotesk-Bold.ttf']:
    fm.fontManager.addfont(f'fonts/Clash_Grotesk/{weight_file}')

plt.rcParams['font.family'] = 'Clash Grotesk'
plt.rcParams['font.weight'] = 'light'

In [ ]:
%%sql -r dataframe_3
USE DATABASE MARCJACOBS_BEAUTY_REVIVAL;

In [ ]:
%%sql -r price_comparison
-- groups by product_type so you compare lipstick to lipstick, eyeliner to eyeliner, etc.
-- keeps only product types that exist in both eras
-- shows average price, average size, and average price_per_oz for each era
-- calculates both absolute and percent change in price_per_oz

WITH comparable_products AS (
  SELECT
    product_type,
    era,
    COUNT(*) AS product_count,
    AVG(price_usd) AS avg_price_usd,
    AVG(size_oz) AS avg_size_oz,
    AVG(price_per_oz) AS avg_price_per_oz
  FROM MARCJACOBS_BEAUTY_REVIVAL.ANALYTICS.PRODUCTS
  WHERE price_per_oz IS NOT NULL
  GROUP BY product_type, era
),
pivoted AS (
  SELECT
    product_type,
    MAX(CASE WHEN era = 'original_launch' THEN product_count END) AS original_product_count,
    MAX(CASE WHEN era = 'revival' THEN product_count END) AS revival_product_count,
    MAX(CASE WHEN era = 'original_launch' THEN avg_price_usd END) AS original_avg_price_usd,
    MAX(CASE WHEN era = 'revival' THEN avg_price_usd END) AS revival_avg_price_usd,
    MAX(CASE WHEN era = 'original_launch' THEN avg_size_oz END) AS original_avg_size_oz,
    MAX(CASE WHEN era = 'revival' THEN avg_size_oz END) AS revival_avg_size_oz,
    MAX(CASE WHEN era = 'original_launch' THEN avg_price_per_oz END) AS original_avg_price_per_oz,
    MAX(CASE WHEN era = 'revival' THEN avg_price_per_oz END) AS revival_avg_price_per_oz
  FROM comparable_products
  GROUP BY product_type
)
SELECT
  product_type,
  original_product_count,
  revival_product_count,
  ROUND(original_avg_price_usd, 2) AS original_avg_price_usd,
  ROUND(revival_avg_price_usd, 2) AS revival_avg_price_usd,
  ROUND(original_avg_size_oz, 2) AS original_avg_size_oz,
  ROUND(revival_avg_size_oz, 2) AS revival_avg_size_oz,
  ROUND(original_avg_price_per_oz, 2) AS original_avg_price_per_oz,
  ROUND(revival_avg_price_per_oz, 2) AS revival_avg_price_per_oz,
  ROUND(revival_avg_price_per_oz - original_avg_price_per_oz, 2) AS price_per_oz_difference,
  ROUND(((revival_avg_price_per_oz / NULLIF(original_avg_price_per_oz, 0)) - 1) * 100, 1) AS pct_change_price_per_oz
FROM pivoted
WHERE original_product_count IS NOT NULL
  AND revival_product_count IS NOT NULL
ORDER BY price_per_oz_difference DESC NULLS LAST, product_type;

In [ ]:
import numpy as np
import pandas as pd

df = price_comparison.to_pandas() if hasattr(price_comparison, 'to_pandas') else price_comparison
df = df.copy()
df.columns = [col.lower() for col in df.columns]
df = df.sort_values('pct_change_price_per_oz', ascending=True)

# --- Chart 1: Diverging bar chart of percent change ---
fig, ax = plt.subplots(figsize=(10, 5))

colors = ['#2a9d8f' if v >= 0 else '#e76f51' for v in df['pct_change_price_per_oz']]
ax.barh(df['product_type'], df['pct_change_price_per_oz'], color=colors,
        edgecolor='#333', linewidth=0.5, height=0.6)
ax.axvline(0, color='black', linewidth=0.8)

for i, (ptype, val) in enumerate(zip(df['product_type'], df['pct_change_price_per_oz'])):
    offset = 3 if val >= 0 else -3
    ha = 'left' if val >= 0 else 'right'
    ax.text(val + offset, i, f'{val:+.1f}%', va='center', ha=ha, fontsize=10, fontweight='medium')

ax.set_xlabel('% change in price per ounce (revival vs original)', fontsize=11, fontweight='normal')
ax.set_title('Percent change by product type', fontsize=13, fontweight='medium')
ax.text(0.5, -0.18,
        'Price-per-ounce comparison for product types in both the 2013 launch and 2026 revival.',
        transform=ax.transAxes, ha='center', fontsize=9, style='italic',
        color='#555', fontweight='normal')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='x', alpha=0.2)
ax.tick_params(labelsize=9)
fig.suptitle('PRICE PER OUNCE CHANGED UNEVENLY IN THE REVIVAL',
             fontsize=18, fontweight='medium', y=0.98)
plt.tight_layout()
plt.show()

# --- Chart 2: Grouped bar chart of absolute price per ounce ---
fig2, ax2 = plt.subplots(figsize=(10, 5))

df_sorted = df.sort_values('original_avg_price_per_oz', ascending=True)
y = np.arange(len(df_sorted))
bar_height = 0.35

ax2.barh(y + bar_height/2, df_sorted['original_avg_price_per_oz'], bar_height,
         label='2013 Collection', color='#264653', edgecolor='#333', linewidth=0.5)
ax2.barh(y - bar_height/2, df_sorted['revival_avg_price_per_oz'], bar_height,
         label='2026 Revival', color='#e9c46a', edgecolor='#333', linewidth=0.5)

for i, (orig, rev) in enumerate(zip(df_sorted['original_avg_price_per_oz'], df_sorted['revival_avg_price_per_oz'])):
    ax2.text(orig + 10, i + bar_height/2, f'${orig:.0f}', va='center', fontsize=9, fontweight='normal')
    ax2.text(rev + 10, i - bar_height/2, f'${rev:.0f}', va='center', fontsize=9, fontweight='normal')

ax2.set_yticks(y)
ax2.set_yticklabels(df_sorted['product_type'], fontsize=10)
ax2.set_xlabel('Average price per ounce ($)', fontsize=11, fontweight='normal')
ax2.set_title('Absolute price per ounce comparison', fontsize=13, fontweight='medium')
ax2.legend(loc='lower right', fontsize=10, frameon=False)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.grid(axis='x', alpha=0.2)
ax2.tick_params(labelsize=9)
fig2.suptitle('ORIGINAL VS REVIVAL: ABSOLUTE PRICE PER OUNCE',
              fontsize=18, fontweight='medium', y=0.98)
plt.tight_layout()
plt.show()

# --- Chart 3: Slope chart of sticker price ---
fig3, ax3 = plt.subplots(figsize=(8, 5))

for _, row in df.iterrows():
    ax3.plot([0, 1], [row['original_avg_price_usd'], row['revival_avg_price_usd']],
            'o-', linewidth=2, markersize=8, alpha=0.8)
    ax3.text(-0.08, row['original_avg_price_usd'], f"${row['original_avg_price_usd']:.0f}",
            va='center', ha='right', fontsize=9, fontweight='normal')
    ax3.text(1.08, row['revival_avg_price_usd'],
            f"{row['product_type']}  ${row['revival_avg_price_usd']:.0f}",
            va='center', ha='left', fontsize=9, fontweight='normal')

ax3.set_xlim(-0.3, 1.4)
ax3.set_xticks([0, 1])
ax3.set_xticklabels(['2013 Collection', '2026 Revival'], fontsize=11, fontweight='normal')
ax3.set_ylabel('Average sticker price ($)', fontsize=11, fontweight='normal')
ax3.set_title('Sticker price barely moved despite unit economics shift', fontsize=13, fontweight='medium')
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)
ax3.grid(axis='y', alpha=0.2)
ax3.tick_params(labelsize=9)
fig3.suptitle('STICKER PRICE VS UNIT ECONOMICS',
              fontsize=18, fontweight='medium', y=0.98)
plt.tight_layout()
plt.show()

print('\nNote: Price per ounce is useful for comparing unit economics, but formats like')
print('eyeliner and mascara can produce extreme values because they are sold in very small quantities.')